<center><img src='https://drive.google.com/uc?export=view&id=12CrUdXDAiltLBT26sG7HZ_HciIhvGyT8'></center>

# Statistical machine learning - Notebook 6, version for students
**Author: Michał Ciach, Dorota Celińska-Kopczyńska, 2025 modifications by BW**  
Exercises denoted with a star \* are optional. They may be more difficult or time-consuming.  

## Description


In today's class, we will analyze the properties of $k$-fold cross validation, the Bootstrap, and feature selection techniques.  


In [ ]:
!pip install --upgrade gdown
!gdown https://drive.google.com/uc?id=1bNShmfSuFvB6gGCjFpRFDGj7Jqes3yRA
!gdown https://drive.google.com/uc?id=1GobUkfo-GR6rmekxgy4gCiHeaCsk9_Ta

Downloading...
From: https://drive.google.com/uc?id=1bNShmfSuFvB6gGCjFpRFDGj7Jqes3yRA
To: /content/8. protein_lengths.tsv
100% 29.3M/29.3M [00:00<00:00, 56.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GobUkfo-GR6rmekxgy4gCiHeaCsk9_Ta
To: /content/8. BDL municipality incomes 2015-2020.csv
100% 228k/228k [00:00<00:00, 4.84MB/s]


## Data & library imports

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import numpy.random as rd
from sklearn.linear_model import LinearRegression
from scipy.stats import ttest_ind, pearsonr, norm
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector
import sklearn.datasets

In [ ]:
protein_lengths = pd.read_csv('8. protein_lengths.tsv', sep='\t')
protein_lengths['LogLength'] = np.log10(protein_lengths['Protein length'])
protein_lengths

,Scientific name,Common name,Protein ID,Protein length,LogLength
0,Homo sapiens,Human,NP_000005.3,1474,3.168497
1,Homo sapiens,Human,NP_000006.2,290,2.462398
2,Homo sapiens,Human,NP_000007.1,421,2.624282
3,Homo sapiens,Human,NP_000008.1,412,2.614897
4,Homo sapiens,Human,NP_000009.1,655,2.816241
...,...,...,...,...,...
648731,Imleria badia,Bay bolete (mushroom),KAF8560453.1,494,2.693727
648732,Imleria badia,Bay bolete (mushroom),KAF8560454.1,737,2.867467
648733,Imleria badia,Bay bolete (mushroom),KAF8560455.1,554,2.743510
648734,Imleria badia,Bay bolete (mushroom),KAF8560456.1,813,2.910091


In [ ]:
human_protein_lengths = protein_lengths.loc[protein_lengths['Common name'] == 'Human'].copy()
# Note: without .copy(), some versions of Pandas may return a View.
# This may interfere with adding a new column to human_protein_lengths.
human_protein_lengths.describe()

,Protein length,LogLength
count,136193.000000,136193.000000
mean,692.655775,2.711540
std,746.993628,0.329892
min,12.000000,1.079181
25%,316.000000,2.499687
50%,514.000000,2.710963
75%,842.000000,2.925312
max,35991.000000,4.556194


In [ ]:
diabetes = sklearn.datasets.load_diabetes(as_frame=True, scaled=False)['frame']
diabetes['sex'] -= 1
diabetes

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,59.0,1.0,32.1,101.00,157.0,93.2,38.0,4.00,4.8598,87.0,151.0
1,48.0,0.0,21.6,87.00,183.0,103.2,70.0,3.00,3.8918,69.0,75.0
2,72.0,1.0,30.5,93.00,156.0,93.6,41.0,4.00,4.6728,85.0,141.0
3,24.0,0.0,25.3,84.00,198.0,131.4,40.0,5.00,4.8903,89.0,206.0
4,50.0,0.0,23.0,101.00,192.0,125.4,52.0,4.00,4.2905,80.0,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,60.0,1.0,28.2,112.00,185.0,113.8,42.0,4.00,4.9836,93.0,178.0
438,47.0,1.0,24.9,75.00,225.0,166.0,42.0,5.00,4.4427,102.0,104.0
439,60.0,1.0,24.9,99.67,162.0,106.6,43.0,3.77,4.1271,95.0,132.0
440,36.0,0.0,30.0,95.00,201.0,125.2,42.0,4.79,5.1299,85.0,220.0


## Estimating the test error of regression with cross-validation

Cross-validation is a technique to estimate the test error of a prediction algorithm (either a classifier or a regression). The idea is to divide the training data set into $k$ parts and, for each part, use it as the test set and the rest as the training set. We will see how well this approach approximates a true test error and what happens for different values of $k$.  

To evaluate models, we typically partition our data set into three subsets: a *training set* on which we train our models, a *validation set* on which we test our models to select the best one, and a *test set* on which we do a final evaluation of our model and, e.g., compare it to a baseline model. One of the approaches to cross-validation is to hold out a portion of the data set as the test set and to split the remaining data into a series of validation sets and training sets.

You should be aware that sometimes people reverse the meaning of the *validation* and *test* sets. This is the case in the picture below, which is taken from Wikipedia—it depicts a series of test sets rather than validation sets. Although the terminology is inconsistent, the general idea stays the same, so if you have a good understanding of how cross-validation works, you won't be confused.


<center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/b/b5/K-fold_cross_validation_EN.svg/1920px-K-fold_cross_validation_EN.svg.png'>

*A diagram representing the k-fold cross-validation.       
Source: https://commons.wikimedia.org/wiki/File:K-fold_cross_validation_EN.svg*
</center>

**Exercise 1.** In this exercise, we will build a linear regression model to predict the progression of diabetes. We will evaluate the model using k-fold cross-validation and check how well the k-fold cross-validation approximates the model's true test error.  

We will use the `diabetes` data frame, which is loaded in the *Data & modules* section. The data frame contains patients' age, sex, BMI (body mass index), bp (average blood pressure), and six blood serum measurements (s1-s6; see [here](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset) for their description). The data frame also contains a measure of the progression of the disease within one year after those measurements were taken (the 'target' column). Let $R$ be the number of rows in this data set.

1. Inspect the `diabetes` data frame. Are there any categorical variables? How are they encoded? Do they all have similar scales (i.e., are they expressed in the same units of measurement, and do they take similar values)?   
  1.1\* Optionally, scale the non-categorical variables so that they have a zero mean and a unit standard deviation.   
  1.2\* Check how scaling influences the model's performance in points 5-9 of this exercise.    
2. Permute the rows of the data frame randomly to eliminate any data grouping that could bias the estimation of the errors. You can use the `sample(frac=1)` command.  
3. Create a data frame with the first $T=50$ observations. This will be our *test set* - our model will need to predict the target variable (i.e., the progression of diabetes) on this data set, but it will not see it during training. The RMSE of prediction on this set is called the *test error*.  
4. Create a data frame with the remaining $R-T$ observations. This will be our *full training set*, where our model will learn how to make predictions. The RMSE of prediction on this set is called the *training error.*  
5. Using the full training data set, create a linear regression model to predict the target variable based on all ten explanatory variables in the data.
Predict the target variable for the test set and calculate the value of the test error.  
  5.1. Is the test error higher or lower than the training error? Should it be? Why?
6. Now, we will implement k-fold cross-validation to predict the test error using only the observations from the training data set. Create a list with several values of the $k$ parameter, including 5, 10 (the two most commonly used values), 15, 60, and the size of the full training set $R-T$ (for a so-called *leave-one-out* cross-validation).  
7. For each value of $k$, perform a $k$-fold cross-validation. For $i=1,\dots, k$:  
  7.1. Select observations $(i-1)\lfloor (R-T)/k \rfloor$ up to $i\lfloor (R-T)/k \rfloor$ from the full training data set and save them as the *validation set*. Save the remaining observations as a *partial training set*.  
  7.2. Create a linear regression model on the partial training set. Calculate its predictions for the validation set. Calculate the validation error (RMSE) and store it in a list.  
8. For each $k$, calculate the mean and standard deviation of the corresponding validation errors.   
  8.1. Can you see some trends in those values? Can you explain it?  
  8.2. Which $k$ best approximates the true test error?
9. Plot the distributions of errors on histograms.   
  9.1. Which $k$ gives you the most information about the distribution of the test error?         
10. In point 7, we have iterated over $i=1,\dots, k$. Do you see a potential problem or inefficiency when $k$ does not divide $R-T$? Does it make sense to iterate over $i=1,\dots, k+1$? What are the possible advantages and disadvantages?

In [ ]:
## Put your code here


True training error: 53.69559881018499
True test error: 52.41396497328264
Cross-validation estimates:
k = 5 mean = 55.8977 sd = 3.7254
k = 10 mean = 55.0742 sd = 5.0212
k = 15 mean = 55.0498 sd = 7.0051
k = 60 mean = 53.5992 sd = 13.6429
k = 392 mean = 44.8523 sd = 32.0824


**Exercise 2.** Let us make a better model! In Exercise 1, we assumed that the effect of s2 (LDL, low-density lipoproteins) and s3 (HDL, high-density lipoproteins) are additive - this means that these variables work "independently" from each other. However, we may reasonably expect a positive interaction between them: if a patient already has a lot of HDL, then any increase in LDL is much worse for them than if the patient had a low HDL. In linear regression, we model that by calculating a product of these variables and treating it as a new variable.

So, including a product of LDL and HDL in our model may increase its accuracy by accounting for interactions between those variables. But you know what? Let us go even further and multiply *every* blood serum variable (s1-s6)! What could possibly go wrong?


1. Write down the mathematical equation for the model described above. Try to keep the equation simple.   
2. Copy the `diabetes` data frame. Add the products of variables s1 to s6 to it (note: we want to multiply each pair only once - do not make a model with both $s_1 s_2$ and $s_2 s_1$).  
  2.1. What would happend if you actually *did* make a model with both $s_1 s_2$ and $s_2 s_1$ variables?
3. Withhold $T=50$ observations (the same ones as in Exercise 1) from the new data frame as the test data set and create a training data set accordingly.  
4. Create a linear regression model on the training data set. Calculate its training and test RMSE. Compare the errors with the model from Exercise 1.  
  4.1. Did the train RMSE decrease? Why?  
  4.2. Did the test RMSE decrease as well? Why?  
  4.3. Is the test RMSE considerably higher than the train RMSE? Is the difference between those errors larger than in Exercise 1?  


In [ ]:
## Put your code here


      age  sex   bmi      bp     s1     s2    s3    s4      s5     s6  ...  \
331  71.0  1.0  24.0   84.00  138.0   85.8  39.0  4.00  4.1897   90.0  ...   
337  54.0  1.0  25.2  115.00  181.0  120.0  39.0  5.00  4.7005   92.0  ...   
313  65.0  1.0  31.3  110.00  213.0  128.0  47.0  5.00  5.2470   91.0  ...   
342  64.0  1.0  28.4  111.00  184.0  127.0  41.0  4.00  4.3820   97.0  ...   
62   41.0  1.0  25.7   83.00  181.0  106.6  66.0  3.00  3.7377   85.0  ...   
..    ...  ...   ...     ...    ...    ...   ...   ...     ...    ...  ...   
272  52.0  0.0  27.0   78.33  134.0   73.0  44.0  3.05  4.4427   69.0  ...   
322  55.0  1.0  32.1  112.67  207.0   92.4  25.0  8.28  6.1048  111.0  ...   
152  39.0  1.0  26.3  115.00  218.0  158.2  32.0  7.00  4.9345  109.0  ...   
177  62.0  0.0  28.9   87.33  206.0  127.2  33.0  6.24  5.4337   99.0  ...   
118  33.0  1.0  25.4  102.00  206.0  141.0  39.0  5.00  4.8675  105.0  ...   

       s2s3      s2s4       s2s5     s2s6    s3s4      s3s5    

**Exercise 3.** Even if a model is reasonable, it may perform poorly if it is too flexible and overfits the data. In the previous exercise, you may (or may not have) concluded that the new model overfits. However, these conclusions could be more robust - after all, we have checked the performance only on a single selected test set. Maybe the result was just a random artifact caused by an unfortunate permutation of the rows of `diabetes` in Exercise 1? That is where cross-validation again comes in handy.    

1. Select your favorite value of $k$ for a k-fold cross-validation.  
2. Use the `cross_val_score` function from the `sklearn` library to evaluate the "complex" model from Exercise 2 and the "simple" model from Exercise 1 (use all observations from `diabetes`). To evaluate a model with RMSE, use a keyword argument `scoring='neg_root_mean_squared_error'`.   
  2.1. Compare the average test errors of the two models. Can you conclude that the complex model overfits?  
  2.2. Check the standard deviation of the errors. Is it much lower than the average of the errors? Is the conclusion about overfitting robust, or could it occur just by chance due to random errors in estimating the mean?      
  2.3. \* Can you come up with some statistical method to decide whether the test RMSE of the complex model is higher than the RMSE of the simple model rather than being caused just by random chance?
3. \* Implement a k-fold cross-validation and, in each iteration, save *both* the training and the test RMSE of the fitted model (or figure out how to get the train RMSE with `sklearn`).   
  3.1 Calculate the mean and standard deviation of the four errors. Analyze the difference between the two models' test MSEs and the train MSEs. Does this provide some additional useful information compared to just the test MSE?  
5. Can the conclusion about overfitting depend on the value of $k$? Why/why not?  


In [ ]:
## Put your code here


Simple model 10-fold MSE mean:	54.63818	 SD:	5.23709
Complex model 10-fold MSE mean:	55.54195	 SD:	5.81463

Validation set size: 44
10-fold cross validation results:
Model 1    	Mean	SD
Train:     	53.3872	0.5372
Validation:	54.7909	4.7534
Model 2    	Mean	SD
Train:     	52.0943	0.682
Validation:	55.964	6.168


## Analyzing estimators with the Bootstrap

*Bootstrap* is a non-parametric technique used to quantify estimation uncertainty and perform hypothesis testing. As you may remember, *non-parametric* means that you do not need to know the probability distribution of your data to use this technique. Therefore, Bootstrap allows you to test hypotheses for any data set, not just normally distributed ones. However, it has the same downsides as all the other non-parametric techniques. In particular, it is difficult to analyze theoretically, so we cannot really say how well it performs by analyzing it, as in the case of the Student's t-test.  

The idea behind Bootstrap is as follows. Suppose we have a statistical sample of size $n$, $X_1, \dots, X_n$, and some statistic (e.g., the mean or the variance) that we compute from this sample, denoted $f(X_1, \dots, X_n)$, to estimate some parameter. Suppose we want to quantify the uncertainty of this estimation but need to know how to calculate the variance of $f(X_1, \dots, X_n)$ analytically. In this case, we can sample $n$ observations **with replacement** from $X_1, \dots, X_n$ to get a new sample $X_1^*, \dots, X_n^*$, calculate $f(X_1^*, \dots, X_n^*)$, and repeat this procedure multiple times to get a sample of values of the statistic $f$. It turns out that the distribution of $f$ obtained this way mimics its true distribution.   

It is important to remember that to get a proper result; we need to get a sample of the same size (otherwise, we change the parameters of our statistic, e.g., the mean computed from a smaller sample will have a higher variance), and to sample with replacement (otherwise, if we sample without replacement, we just get a permutation of the values).

**Exercise 4.** In this exercise, we will use the bootstrap technique to analyze the properties of the estimator of the standard deviation of a protein log-length.

1. Select $N=50$ random samples from the `human_protein_lengths` data and put them into a new data frame called `sample`. Calculate the standard deviation of the sampled protein log-lengths.  
2. Now, we will use the bootstrap to estimate the distribution of the estimator. Repeat the following $R=10 000$ times:  
  2.1. Resample the observations from the `sample` data frame, e.g., using the `sample.sample()` method.  
  2.2. Calculate the standard deviation from the resampled observations and store it in a list.   
3. Plot the obtained bootstrap replicates of the standard deviation on a histogram. Calculate the estimator's mean and standard deviation and check its coefficient of variation.    
  3.1. In your opinion, is the standard deviation estimated sufficiently precisely ?
4. Estimate the true distribution of the standard deviation estimator. To do this, draw $R$ samples of size $N$ of the log-lengths of human proteins and calculate the standard deviation for each sample.  
  4.1. Plot the estimator's obtained $R$ values on a histogram and compare it to the histogram of bootstrapped replicates. Are the distributions similar?  
  4.2. Use the $R$ values to estimate the estimator's expected value and standard deviation. Compare it to the values obtained with the bootstrap. Are the bootstrap estimates similar to the true parameters of this estimator?  


In [ ]:
## Put your code here


True std: 0.32989112244952207
Sampled mean: 2.671060777582481
Sampled std: 0.30433457055553276
Bootstrapped mean and std of stds: 0.3003 0.0271
Bootstrapped CV of the estimator of the std: 0.0901
True mean and std of stds: 0.3244 0.0351
True CV of the estimator of the std: 0.1083
